In [11]:
# IMPORTS

import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from tensorflow.keras.models import load_model
from pytorch_tabnet.tab_model import TabNetClassifier

import warnings
warnings.filterwarnings("ignore")


In [12]:
# PATHS

root_path = Path.cwd().resolve()
if root_path.name == "notebooks":
    root_path = root_path.parent

data_dir = root_path / "data"
artifact_dir = root_path / "artifacts"
model_dir = root_path / "models"
result_dir = root_path / "results"

for directory in [data_dir, artifact_dir, model_dir, result_dir]:
    directory.mkdir(parents=True, exist_ok=True)

raw_data_path = data_dir / "downsampled_transactions.csv"


In [13]:
# PREPROCESSING HELPERS

FEATURE_COLS = [
    "log_amount",
    "error_orig",
    "error_dest",
    "hour",
    "is_night",
    "is_high_amount",
    "type_encoded",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

TYPE_MAPPING = {"TRANSFER": 0, "CASH_OUT": 1}
RAW_NUMERIC_COLS = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]


def clean_raw_transactions(df):
    df = df.copy()

    for col in RAW_NUMERIC_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "type" not in df.columns:
        raise ValueError("Missing required column: type")

    if "isFraud" in df.columns:
        df["isFraud"] = pd.to_numeric(df["isFraud"], errors="coerce").astype("Int64")

    df = df.dropna(subset=["type"] + RAW_NUMERIC_COLS)
    return df


def _binary_indicator(series):
    numeric = pd.to_numeric(series, errors="coerce")
    text = series.astype(str).str.strip().str.lower()
    return numeric.fillna(text.isin(["true", "yes", "1"]).astype(int)).astype(int)


def engineer_features(df, high_amount_threshold, type_mapping=None):
    df = clean_raw_transactions(df)
    type_mapping = TYPE_MAPPING if type_mapping is None else type_mapping

    df = df[df["type"].isin(type_mapping)].copy()

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
    df["error_orig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    df["error_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    df["hour"] = (df["step"] % 24).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    df["is_high_amount"] = (df["amount"] > high_amount_threshold).astype(int)
    df["type_encoded"] = df["type"].map(type_mapping).astype(int)

    if "nameDest" in df.columns:
        df["is_merchant_dest"] = df["nameDest"].astype(str).str.startswith("M").astype(int)
    elif "is_merchant_dest" in df.columns:
        df["is_merchant_dest"] = _binary_indicator(df["is_merchant_dest"])
    else:
        df["is_merchant_dest"] = 0

    for col in FEATURE_COLS + ["is_merchant_dest"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


In [14]:
# READ BEST MODEL METADATA

best_model_metadata_path = model_dir / "best_model_metadata.json"
if not best_model_metadata_path.exists():
    raise FileNotFoundError(
        "Missing models/best_model_metadata.json. "
        "Run result_analysis.ipynb after model_training.ipynb before inference."
    )

with open(best_model_metadata_path, "r", encoding="utf-8") as f:
    best_model_metadata = json.load(f)

best_model_name = best_model_metadata["model_name"]
model_family = best_model_metadata["model_family"]
model_path = root_path / best_model_metadata["artifact_path"]

print("Best Model:", best_model_name)

Best Model: MLP


In [15]:
# LOAD BEST MODEL AND PREPROCESSING

preprocessing_bundle_path = artifact_dir / "preprocessing.pkl"
if not preprocessing_bundle_path.exists():
    raise FileNotFoundError("Missing artifacts/preprocessing.pkl. Run model_training.ipynb before inference.")

with open(preprocessing_bundle_path, "rb") as f:
    preprocessing_bundle = pickle.load(f)

feature_cols = preprocessing_bundle["feature_cols"]
metadata = preprocessing_bundle["metadata"]
scaler = preprocessing_bundle["scaler"]
decision_thresholds = preprocessing_bundle.get("decision_thresholds", {}).copy()
if best_model_name not in decision_thresholds and best_model_metadata.get("decision_threshold"):
    decision_thresholds[best_model_name] = best_model_metadata["decision_threshold"]

if not model_path.exists():
    raise FileNotFoundError(f"Missing best model artifact for {best_model_name}: {model_path}. Run result_analysis.ipynb before inference.")

if model_family == "keras":
    model = load_model(model_path)

elif model_family == "tabnet":
    model = TabNetClassifier()
    model.load_model(str(model_path))

else:
    raise ValueError(f"Unsupported model family: {model_family}")

print(f"{best_model_name} loaded successfully!")
print("Feature columns:", feature_cols)
print("Decision threshold:", decision_thresholds.get(best_model_name, {}).get("threshold", 0.5))


MLP loaded successfully!
Feature columns: ['log_amount', 'error_orig', 'error_dest', 'hour', 'is_night', 'is_high_amount', 'type_encoded', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Decision threshold: 0.49446097016334534


In [16]:
# LOAD DATA

if not raw_data_path.exists():
    raise FileNotFoundError("Missing data/downsampled_transactions.csv. Run preprocessing_pipeline.ipynb before inference.")

df = pd.read_csv(raw_data_path)


In [17]:
# FRAUD PREDICTION FUNCTION

def prepare_model_input(input_data):
    engineered = engineer_features(
        input_data,
        metadata["high_amount_threshold"],
        metadata["type_mapping"],
    )

    if engineered.empty:
        raise ValueError("Transaction type must be TRANSFER or CASH_OUT.")

    X = engineered.reindex(columns=feature_cols, fill_value=0)
    X = scaler.transform(X).astype("float32")
    return X


def predict_transaction(input_data, model, best_model_name):
    X = prepare_model_input(input_data)

    if best_model_name in {"GRU", "BiLSTM"}:
        X_model = X.reshape((X.shape[0], 1, X.shape[1]))
    else:
        X_model = X

    if best_model_name == "TabNet":
        prob = float(model.predict_proba(X_model)[0, 1])
    else:
        prob = float(model.predict(X_model, verbose=0).ravel()[0])

    threshold = decision_thresholds.get(best_model_name, {}).get("threshold", 0.5)
    pred = int(prob >= threshold)
    return pred, prob, threshold


In [18]:
# DATASET ROW TEST

row_number = 90
sample = df.iloc[[row_number]].copy()
actual = sample["isFraud"].iloc[0]

prediction, probability, threshold = predict_transaction(
    sample,
    model,
    best_model_name,
)

print("===== DATASET TEST =====")
print("Actual      :", bool(actual))
print("Predicted   :", bool(prediction))
print("Probability :", float(probability))
print("Threshold   :", float(threshold))


===== DATASET TEST =====
Actual      : False
Predicted   : False
Probability : 0.005291522014886141
Threshold   : 0.49446097016334534


In [19]:
# MANUAL TRANSACTION TEST

manual_transaction = {
    "step": 137,
    "type": "CASH_OUT",
    "amount": 50000,
    "oldbalanceOrg": 200000,
    "newbalanceOrig": 150000,
    "oldbalanceDest": 1000000,
    "newbalanceDest": 1050000,
}

manual_df = pd.DataFrame([manual_transaction])

prediction, probability, threshold = predict_transaction(
    manual_df,
    model,
    best_model_name,
)

print("===== MANUAL TEST =====")
print("Fraud Prediction :", bool(prediction))
print("Probability      :", float(probability))
print("Threshold        :", float(threshold))


===== MANUAL TEST =====
Fraud Prediction : False
Probability      : 0.0019802586175501347
Threshold        : 0.49446097016334534
